# 03 — ML-begreper uten matte

**Fase:** 1 — Fundamenter | **Tid:** 1–2 timer | **Krav:** Notatbok 01–02

**Hva du bygger:** En intuitiv forståelse av maskinlæring — ordforrådet og konseptene du trenger for å jobbe med LLMs og AI-systemer, uten å skrive en eneste treningsfunksjon.

---

## Hva er maskinlæring, egentlig?

Tradisjonell programmering: Du skriver **regler** → datamaskinen følger dem.  
Maskinlæring: Du gir **eksempler** → datamaskinen finner reglene selv.

```
Tradisjonell:  Input + Regler → Output
ML:            Input + Output → Regler (= modellen)
```

En **modell** er en matematisk funksjon med millioner av parametre (tall) som ble justert under trening til å gjøre noe nyttig — f.eks. forutsi neste ord i en setning.

In [ ]:
# Enkleste mulige 'modell': en lineær funksjon
# Under trening justeres vekt (w) og bias (b) automatisk

def modell(x, w=2.5, b=1.0):
    """y = w*x + b  (en linje med stigningstall w og skjæringspunkt b)"""
    return w * x + b

# Treningsdata: [input, forventet_output]
treningsdata = [(1, 3.5), (2, 6.0), (3, 8.5), (4, 11.0)]

for x, fasit in treningsdata:
    prediksjon = modell(x)
    feil = fasit - prediksjon
    print(f"x={x}  fasit={fasit}  modell={prediksjon:.1f}  feil={feil:.1f}")

print("\nMålet med trening: minimere feil ved å justere w og b automatisk.")

---

## Trening vs. Inferens

To helt forskjellige faser — viktig å holde dem fra hverandre:

| | **Trening** | **Inferens** |
|-|-------------|-------------|
| Hva skjer | Modellen lærer av data | Modellen brukes |
| Hvem gjør det | ML-forskere (ofte uker/måneder) | Deg (millisekunder per kall) |
| Kostnad | Svært dyrt (GPU-klynger) | Billig per kall |
| Eksempel | OpenAI trener GPT-4 | Du kaller `client.chat.completions.create(...)` |

**Som AI Engineer jobber du nesten utelukkende med inferens** — du bruker ferdigtrente modeller via API. Trening overlater du til ML-forskere.

---

## Tokens — LLMs leser ikke ord

LLMs (store språkmodeller) deler tekst opp i **tokens** — biter som er kortere enn ord, lengre enn bokstaver.

- `"hei"` → 1 token  
- `"pensjon"` → 1 token  
- `"pensjonsordning"` → 2–3 tokens  
- `"superintelligens"` → 3–4 tokens

**Hvorfor betyr dette noe?**  
- Du betaler per token (både input og output)  
- Modellen har en **kontekstvindu**-grense (f.eks. 128 000 tokens for GPT-4o)  
- Lange dokumenter må deles opp (chunking) for å passe

In [ ]:
%pip install -q tiktoken

In [ ]:
import tiktoken

# tiktoken er OpenAIs tokenizer — viser deg nøyaktig hva GPT-4 "ser"
enc = tiktoken.encoding_for_model("gpt-4o")

tekster = [
    "Hei",
    "Pensjon",
    "AFP gir deg rett til å gå av tidlig med pensjon fra avtalt pensjonsalder.",
    "Statens pensjonskasse forvalter pensjonsordningen for statsansatte.",
]

for tekst in tekster:
    tokens = enc.encode(tekst)
    token_tekster = [enc.decode([t]) for t in tokens]
    print(f"'{tekst[:40]}' → {len(tokens)} tokens: {token_tekster}")

In [ ]:
# Beregn kostnad for en LLM-forespørsel
PRIS_PER_1K_INPUT_TOKENS  = 0.0025  # USD, GPT-4o (omtrentlig)
PRIS_PER_1K_OUTPUT_TOKENS = 0.010

prompt = """Du er en pensjonsrådgiver. Forklar AFP på en enkel måte."""
svar   = "AFP, eller avtalefestet pensjon, lar deg gå av med pensjon fra 62 år."

input_tokens  = len(enc.encode(prompt))
output_tokens = len(enc.encode(svar))

kostnad = (input_tokens / 1000 * PRIS_PER_1K_INPUT_TOKENS +
           output_tokens / 1000 * PRIS_PER_1K_OUTPUT_TOKENS)

print(f"Input:  {input_tokens} tokens")
print(f"Output: {output_tokens} tokens")
print(f"Kostnad per kall: ${kostnad:.6f} USD")
print(f"Kostnad for 10 000 kall: ${kostnad * 10_000:.2f} USD")

---

## Transformeren — arkitekturen bak alle LLMs

Alle store språkmodeller (GPT-4, Claude, Gemini) er varianter av **Transformer**-arkitekturen (publisert 2017).

### Kjerneideen: Attention

Når modellen leser `"AFP gir deg rett til å gå av tidlig"`, beregner den for hvert ord **hvor mye det skal bry seg om hvert annet ord**.

```
"tidlig" ser på:
  - "gå av"  → høy vekt (semantisk koblet)
  - "AFP"    → middels vekt
  - "til"    → lav vekt
```

Dette gir modellen kontekstforståelse — den vet at `"bank"` betyr noe annet i `"gå til banken"` vs `"bratt elveskrent"`.

In [ ]:
import numpy as np

# Forenklet illustrasjon av attention (IKKE den faktiske formelen, men intuisjonen)
ord = ["AFP", "gir", "deg", "rett", "til", "tidlig"]

# Simuler attention-vekter: hva "tidlig" ser på
np.random.seed(42)
vekter = np.array([0.35, 0.05, 0.10, 0.20, 0.08, 0.22])  # summer til 1.0

print("Attention fra 'tidlig' til hvert ord:")
for ord_, vekt in zip(ord, vekter):
    søyle = "█" * int(vekt * 40)
    print(f"  {ord_:8s} {søyle} {vekt:.2f}")

---

## Viktige begreper du vil møte

| Begrep | Forklaring | Analogie |
|--------|-----------|----------|
| **Parameter** | Justerbare tall i modellen | Knapper på et kontrollpanel |
| **Loss/tap** | Mål på hvor feil modellen er | Score du vil minimere |
| **Epoch** | Én gjennomgang av hele treningsdatasettet | Én lesesessjon av pensjonsloven |
| **Batch** | Delmengde av data behandlet om gangen | En bunke dokumenter |
| **Overfitting** | Modellen memorerer trening, fungerer dårlig på nytt | Pugget fasiten, ikke forstått |
| **Fine-tuning** | Videretrenig av en ferdigtrent modell | Videreutdanning |
| **Inferens** | Bruke modellen til å generere svar | Bruke kunnskapen |
| **Latens** | Tid fra spørsmål til svar | Responstid |
| **Kontekstvindu** | Maks tokens modellen kan "huske" | Arbeidsminne |
| **Temperature** | Hvor kreativ/tilfeldig modellen er (0–2) | 0=forutsigbar, 2=kreativ |

In [ ]:
# Illustrer temperature: hva betyr det i praksis?
import random

def velg_neste_ord(kandidater: dict, temperature: float) -> str:
    """Forenklet: høy temperature = mer tilfeldig valg."""
    if temperature == 0:
        return max(kandidater, key=kandidater.get)  # Alltid mest sannsynlig
    
    # Skaler sannsynlighetene med temperature
    import math
    justerte = {k: math.exp(v / temperature) for k, v in kandidater.items()}
    total = sum(justerte.values())
    normalisert = {k: v / total for k, v in justerte.items()}
    
    return random.choices(list(normalisert.keys()),
                          weights=list(normalisert.values()))[0]

# Sannsynligheter modellen har beregnet for neste ord etter "AFP er en ..."
kandidater = {"pensjonsordning": 0.6, "rettighet": 0.25, "forkortelse": 0.1, "drøm": 0.05}

random.seed(42)
for temp in [0, 0.3, 0.7, 1.5]:
    valg = [velg_neste_ord(kandidater, temp) for _ in range(10)]
    fra_valg = {}
    for v in valg:
        fra_valg[v] = fra_valg.get(v, 0) + 1
    print(f"Temperature={temp}: {fra_valg}")

---

## Supervised vs. Unsupervised vs. Reinforcement Learning

```
Supervised:     Lærer av merkede eksempler  → "dette bildet er en katt"
Unsupervised:   Finner mønstre i umerkede data → clustering, embeddings
Reinforcement:  Lærer av belønning/straff → ChatGPT RLHF-trening
```

**LLMs bruker alle tre:**
1. **Supervised** — forutsi neste token (selvmerking: teksten er sin egen fasit)
2. **Unsupervised** — lære representasjoner (embeddings) fra rå tekst
3. **Reinforcement (RLHF)** — menneskelige tilbakemeldinger gjør svarene bedre

---

## Oppsummering

Du vet nå:
- Hva en modell er og forskjellen mellom trening og inferens
- Hva tokens er og hvorfor de koster penger
- Hva Transformer-arkitekturen og attention gjør (intuitivt)
- De viktigste ML-begrepene du vil møte som AI Engineer

---

## Hva er neste steg?

**Neste notatbok: `04_llm_fundamentals.ipynb`**  
Nå som du har begrepene på plass, dykker vi dypere inn i LLMs: tokenisering med tiktoken, kontekstvinduer i praksis, og hva som faktisk skjer inne i modellen når du sender en melding.